# SQL-Einstieg

In diesem Notebook lernt ihr die wichtigsten SQL-Befehle an einer kleinen Bücherei-Datenbank kennen. Die Begriffe dazu stehen in [grundlagen.md](grundlagen.md).

So funktioniert es:

- **Beispiele** sind fertig. Führt sie aus und schaut euch das Ergebnis an.
- **Übungen** löst ihr selbst in der Zelle darunter. Ersetzt dort die Zeile `SELECT 'TODO: ...'` durch eure Abfrage. Unter jeder Übung steht das **erwartete Ergebnis**, damit ihr euch selbst kontrollieren könnt. Die Reihenfolge der Zeilen darf abweichen, außer die Aufgabe verlangt eine Sortierung.
- Wenn etwas schiefgeht: Kernel neu starten (**Restart**) und alle Zellen von oben ausführen (**Run All**). Die Datenbank wird dabei neu aufgebaut.

# Setup

Diese Zelle lädt die SQL-Erweiterung für Jupyter und öffnet die Datenbankdatei `buecherei.db` in diesem Ordner. Gibt es sie noch nicht, legt SQLite sie an. Ansehen könnt ihr sie mit der VS-Code-Erweiterung *SQLite Viewer*. `PRAGMA foreign_keys = ON` schaltet die Prüfung von Fremdschlüsseln ein, die bei SQLite sonst aus ist.

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 20
%sql sqlite:///buecherei.db
%sql PRAGMA foreign_keys = ON;

# 1. Tabellen anlegen

Die Bücherei hat drei Tabellen:

- `autor`: wer Bücher schreibt
- `buch`: jedes Buch gehört zu genau einem Autor (Fremdschlüssel `autor_id`)
- `ausleihe`: wer welches Buch wann ausgeliehen hat. Ist `zurueck_am` leer (`NULL`), ist das Buch noch ausgeliehen.

`DROP TABLE IF EXISTS` löscht eine Tabelle, falls es sie schon gibt. So lässt sich die Zelle beliebig oft ausführen.

In [ ]:
%%sql
DROP TABLE IF EXISTS ausleihe;
DROP TABLE IF EXISTS buch;
DROP TABLE IF EXISTS autor;

CREATE TABLE autor (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    geburtsjahr INTEGER
);

CREATE TABLE buch (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    titel TEXT NOT NULL,
    erscheinungsjahr INTEGER NOT NULL,
    seiten INTEGER NOT NULL CHECK (seiten > 0),
    autor_id INTEGER NOT NULL,
    FOREIGN KEY (autor_id) REFERENCES autor(id)
);

CREATE TABLE ausleihe (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    buch_id INTEGER NOT NULL,
    mitglied TEXT NOT NULL,
    ausgeliehen_am DATE NOT NULL,
    zurueck_am DATE,
    FOREIGN KEY (buch_id) REFERENCES buch(id)
);

# 2. Daten einfügen

Bei `INSERT` lassen wir die `id` weg, die vergibt die Datenbank selbst (`AUTOINCREMENT`). Texte stehen in einfachen Anführungszeichen, Zahlen nicht.

In [ ]:
%%sql
INSERT INTO autor (name, geburtsjahr) VALUES
('Mara Lind', 1975),
('Jonas Berg', 1982),
('Ella Stein', 1990),
('Theo Krämer', 1968);

INSERT INTO buch (titel, erscheinungsjahr, seiten, autor_id) VALUES
('Der leise Hafen', 2015, 280, 1),
('Nebel über Kerrin', 2018, 342, 1),
('Das Uhrwerk', 2020, 198, 2),
('Sternenstaub', 2012, 410, 2),
('Die letzte Fähre', 2021, 256, 3),
('Kalter Sand', 2019, 305, 1),
('Das Gartenhaus', 2023, 174, 3),
('Funkstille', 2016, 230, 2);

INSERT INTO ausleihe (buch_id, mitglied, ausgeliehen_am, zurueck_am) VALUES
(1, 'Kim', '2025-09-01', '2025-09-20'),
(3, 'Sam', '2025-09-05', NULL),
(1, 'Lou', '2025-10-02', '2025-10-15'),
(5, 'Kim', '2025-10-10', NULL),
(4, 'Noa', '2025-10-12', '2025-11-01'),
(2, 'Sam', '2025-11-03', NULL);

# 3. Daten abfragen: SELECT

### Beispiel: Alle Spalten aller Autoren
`*` heißt „alle Spalten“.

In [ ]:
%%sql
SELECT * FROM autor;

### Beispiel: Nur bestimmte Spalten

In [ ]:
%%sql
SELECT titel, seiten FROM buch;

### Übung 3.1
Gebt Titel und Erscheinungsjahr aller Bücher aus.

**Erwartetes Ergebnis:**

| titel | erscheinungsjahr |
|---|---|
| Der leise Hafen | 2015 |
| Nebel über Kerrin | 2018 |
| Das Uhrwerk | 2020 |
| Sternenstaub | 2012 |
| Die letzte Fähre | 2021 |
| Kalter Sand | 2019 |
| Das Gartenhaus | 2023 |
| Funkstille | 2016 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

# 4. Zeilen filtern: WHERE

### Beispiel: Bücher ab 2019
Vergleiche: `=`, `<>` (ungleich), `<`, `<=`, `>`, `>=`. Mehrere Bedingungen verbindet man mit `AND` und `OR`.

In [ ]:
%%sql
SELECT titel, erscheinungsjahr FROM buch
WHERE erscheinungsjahr >= 2019;

### Übung 4.1
Welche Bücher haben mehr als 300 Seiten? Gebt Titel und Seitenzahl aus.

**Erwartetes Ergebnis:**

| titel | seiten |
|---|---|
| Nebel über Kerrin | 342 |
| Sternenstaub | 410 |
| Kalter Sand | 305 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 4.2
Welche Bücher beginnen mit „Das“?

*Tipp: `LIKE 'Das %'` findet Texte, die mit „Das “ beginnen. `%` steht für beliebig viele Zeichen.*

**Erwartetes Ergebnis:**

| titel |
|---|
| Das Uhrwerk |
| Das Gartenhaus |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 4.3
Welche Ausleihen sind noch nicht zurückgegeben? Gebt `buch_id` und `mitglied` aus.

*Tipp: Auf `NULL` prüft man mit `IS NULL`, nicht mit `= NULL`.*

**Erwartetes Ergebnis:**

| buch_id | mitglied |
|---|---|
| 3 | Sam |
| 5 | Kim |
| 2 | Sam |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

# 5. Sortieren und begrenzen: ORDER BY und LIMIT

### Beispiel: Bücher nach Seitenzahl, die dicksten zuerst
`ASC` sortiert aufsteigend (Standard), `DESC` absteigend.

In [ ]:
%%sql
SELECT titel, seiten FROM buch
ORDER BY seiten DESC;

### Übung 5.1
Gebt die 3 neuesten Bücher mit Titel und Erscheinungsjahr aus, das neueste zuerst.

**Erwartetes Ergebnis:**

| titel | erscheinungsjahr |
|---|---|
| Das Gartenhaus | 2023 |
| Die letzte Fähre | 2021 |
| Das Uhrwerk | 2020 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

# 6. Zusammenfassen: COUNT, AVG, MIN, MAX, SUM, GROUP BY

### Beispiel: Anzahl Bücher und durchschnittliche Seitenzahl
`AS` gibt einer Ergebnisspalte einen Namen.

In [ ]:
%%sql
SELECT COUNT(*) AS anzahl, AVG(seiten) AS durchschnitt FROM buch;

### Beispiel: Anzahl Ausleihen pro Mitglied
`GROUP BY` bildet eine Gruppe pro Mitglied. `COUNT(*)` zählt dann pro Gruppe.

In [ ]:
%%sql
SELECT mitglied, COUNT(*) AS ausleihen
FROM ausleihe
GROUP BY mitglied;

### Übung 6.1
Wie viele Seiten hat das dickste Buch?

**Erwartetes Ergebnis:**

| max_seiten |
|---|
| 410 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 6.2
Wie viele Bücher gibt es pro `autor_id`?

**Erwartetes Ergebnis:**

| autor_id | anzahl |
|---|---|
| 1 | 3 |
| 2 | 3 |
| 3 | 2 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 6.3
Welche `autor_id` hat mehr als 2 Bücher?

*Tipp: `WHERE` filtert Zeilen **vor** dem Gruppieren, `HAVING` filtert Gruppen **danach**.*

**Erwartetes Ergebnis:**

| autor_id | anzahl |
|---|---|
| 1 | 3 |
| 2 | 3 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

# 7. Tabellen verbinden: JOIN

Bisher stand jede Abfrage auf **einer** Tabelle. Die Titel stehen aber in `buch`, die Namen in `autor`. Wie bringt man beides zusammen?

### Ausprobieren: Zwei Tabellen in `FROM`
Ihr wollt zu jedem Buch den Namen des Autors sehen. Erste Idee: einfach beide Tabellen hinter `FROM` schreiben, getrennt durch ein Komma. Führt die Zelle aus und schaut genau hin.

In [ ]:
%%sql
SELECT buch.titel, autor.name
FROM buch, autor;

### Übung 7.1: Was ist da passiert?
Stimmt die Zeile „Der leise Hafen | Jonas Berg“? Und warum steht unter dem Ergebnis, dass es gekürzt wurde?

Zählt mit `COUNT(*)`, wie viele Zeilen die Abfrage oben **wirklich** liefert. Beantwortet danach diese Fragen (Doppelklick auf die Zelle zum Bearbeiten):

1. Wie viele Zeilen haben `buch` und `autor` jeweils?
2. Wie hängen diese beiden Zahlen mit eurem Ergebnis zusammen?
3. Wie viele der Zeilen sind „richtig“?

*Antworten:* ...

**Erwartetes Ergebnis:**

| anzahl |
|---|
| 32 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Das kartesische Produkt
Ohne Bedingung kombiniert SQL **jede** Zeile der einen Tabelle mit **jeder** Zeile der anderen. Das nennt man **kartesisches Produkt** (nach dem Mathematiker René Descartes, lateinisch Cartesius).

8 Bücher × 4 Autoren = 32 Zeilen. Nur 8 davon gehören wirklich zusammen, der Rest ist Unsinn.

Mit jeder weiteren Tabelle wird weiter multipliziert: Nimmt man `ausleihe` (6 Zeilen) dazu, sind es schon 8 × 4 × 6 = 192 Zeilen.

### Übung 7.2: Die richtigen Paare herausfiltern
Nehmt die Abfrage von oben und behaltet mit `WHERE` nur die Zeilen, in denen Buch und Autor wirklich zusammengehören.

*Tipp: Welche Spalte in `buch` zeigt auf welche Spalte in `autor`?*

**Erwartetes Ergebnis:**

| titel | name |
|---|---|
| Der leise Hafen | Mara Lind |
| Nebel über Kerrin | Mara Lind |
| Das Uhrwerk | Jonas Berg |
| Sternenstaub | Jonas Berg |
| Die letzte Fähre | Ella Stein |
| Kalter Sand | Mara Lind |
| Das Gartenhaus | Ella Stein |
| Funkstille | Jonas Berg |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### JOIN: dasselbe, nur übersichtlicher
Genau das macht `JOIN ... ON`. Man kann sich einen JOIN als kartesisches Produkt vorstellen, von dem nur die Zeilen übrig bleiben, die die `ON`-Bedingung erfüllen. Die Datenbank baut dafür intern nicht wirklich alle Kombinationen auf, sondern rechnet geschickter. Das Ergebnis ist aber dasselbe.

Warum dann `JOIN`? Die Verbindung steht direkt bei der Tabelle, `WHERE` bleibt frei für echte Filter, und eine fehlende Bedingung fällt schneller auf.

### Beispiel: Titel mit Namen des Autors
Dieselbe Abfrage wie in Übung 7.2, diesmal mit `JOIN ... ON`. Vergleicht die Ergebnisse.

In [ ]:
%%sql
SELECT buch.titel, autor.name
FROM buch
JOIN autor ON autor.id = buch.autor_id;

### Übung 7.3
Welche Bücher hat Mara Lind geschrieben?

**Erwartetes Ergebnis:**

| titel |
|---|
| Der leise Hafen |
| Nebel über Kerrin |
| Kalter Sand |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 7.4
Wie viele Bücher haben die Autoren geschrieben? Gebt **Name** und Anzahl aus, sortiert nach Anzahl absteigend und bei Gleichstand nach Name. Autoren ohne Buch müssen hier noch nicht erscheinen, um die geht es in Übung 7.5.

**Erwartetes Ergebnis:**

| name | anzahl |
|---|---|
| Jonas Berg | 3 |
| Mara Lind | 3 |
| Ella Stein | 2 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Beispiel: Auch Autoren ohne Buch anzeigen: LEFT JOIN
`LEFT JOIN` behält **alle** Zeilen der linken Tabelle. Gibt es keinen Partner, stehen rechts `NULL`-Werte.

In [ ]:
%%sql
SELECT autor.name, buch.titel
FROM autor
LEFT JOIN buch ON buch.autor_id = autor.id;

### Übung 7.5
Welcher Autor hat kein Buch?

*Tipp: `LEFT JOIN` und dann auf `IS NULL` filtern.*

**Erwartetes Ergebnis:**

| name |
|---|
| Theo Krämer |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 7.6
Welche Bücher wurden noch nie ausgeliehen?

**Erwartetes Ergebnis:**

| titel |
|---|
| Kalter Sand |
| Das Gartenhaus |
| Funkstille |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 7.7
Welche Bücher sind gerade ausgeliehen und von wem? Gebt Titel und Mitglied aus.

**Erwartetes Ergebnis:**

| titel | mitglied |
|---|---|
| Das Uhrwerk | Sam |
| Die letzte Fähre | Kim |
| Nebel über Kerrin | Sam |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Wenn man jede Kombination wirklich will: CROSS JOIN
Manchmal ist das kartesische Produkt genau richtig. Die Bücherei will jedes Buch in zwei Formaten anbieten: als Taschenbuch und als Hörbuch. Die Formate stehen in einer neuen Tabelle.

Mit `CROSS JOIN` schreibt man ausdrücklich hin: „Ja, ich will jede Kombination.“ So weiß auch jeder, der die Abfrage später liest, dass das Absicht ist.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS format (name TEXT PRIMARY KEY);
DELETE FROM format;
INSERT INTO format (name) VALUES ('Taschenbuch'), ('Hörbuch');

SELECT buch.titel, format.name AS format
FROM buch
CROSS JOIN format
ORDER BY buch.titel, format.name;

### Übung 7.8
Ein drittes Format kommt dazu: `E-Book`. Fügt es in die Tabelle `format` ein. Wie viele Zeilen liefert der `CROSS JOIN` jetzt? **Erst im Kopf rechnen**, dann mit `COUNT(*)` prüfen.

**Erwartetes Ergebnis:**

| anzahl |
|---|
| 24 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Merke

| Ihr schreibt | Ergebnis |
|---|---|
| `FROM a, b` oder `FROM a CROSS JOIN b` | kartesisches Produkt: Zeilen von `a` × Zeilen von `b` |
| `FROM a JOIN b ON ...` | nur die Kombinationen, die die Bedingung erfüllen |

Hat ein Ergebnis verdächtig viele Zeilen, ist fast immer eine Join-Bedingung vergessen worden. Ein typischer gewollter Einsatz: jedem Charakter in einem Spiel jeden Wert (HP, Mana, …) mit einem Startwert geben. In Teil 3 seht ihr, was eine vergessene Bedingung bei großen Tabellen kostet.

# 8. Abfragen in Abfragen

### Beispiel: Bücher, die dicker sind als der Durchschnitt
Die Abfrage in Klammern wird zuerst ausgeführt und liefert einen einzelnen Wert.

In [ ]:
%%sql
SELECT titel, seiten FROM buch
WHERE seiten > (SELECT AVG(seiten) FROM buch);

### Übung 8.1
Welches Buch ist das älteste? Löst die Aufgabe **ohne** `LIMIT`, mit einer Abfrage in der Abfrage.

**Erwartetes Ergebnis:**

| titel | erscheinungsjahr |
|---|---|
| Sternenstaub | 2012 |

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

# 9. Daten ändern

Ab hier ändert ihr die Daten. Die erwarteten Ergebnisse oben passen danach nicht mehr. Wenn ihr oben noch einmal üben wollt: **Restart** und **Run All**.

### Beispiel: Ein Buch ändern
**Achtung:** Ohne `WHERE` ändert `UPDATE` **alle** Zeilen. Dasselbe gilt für `DELETE`.

In [ ]:
%%sql
UPDATE buch SET seiten = 200 WHERE id = 3;
SELECT * FROM buch WHERE id = 3;

### Übung 9.1
Fügt ein neues Buch von Ella Stein ein: „Wellenbrecher“, erschienen 2024, 220 Seiten. Prüft danach mit einem `SELECT`, ob es da ist.

*Tipp: Welche `id` hat Ella Stein?*

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 9.2
Sam hat „Das Uhrwerk“ am 2025-11-10 zurückgegeben. Tragt das Datum bei der passenden Ausleihe ein.

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

### Übung 9.3: Regeln testen
Versucht, ein Buch mit `autor_id = 99` einzufügen, und danach eines mit `-5` Seiten.

**Beide Versuche müssen mit einer roten Fehlermeldung scheitern.** Das ist hier gewollt: Die Datenbank schützt sich selbst vor ungültigen Daten. Welche Meldung kommt jeweils?

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

In [ ]:
%%sql
SELECT 'TODO: Hier eure Abfrage schreiben' AS aufgabe;

# Geschafft!

Weiter geht es mit [Teil 1: Anomalien und Normalformen](../1-normalisierung/Anforderungen.md).